# Experiment 1 — Final reporting (Phase 6)

Scores the frozen 20 primary approaches plus the 8 predefined seed repeats on
the held-out test split, once. Requires the frozen Phase 4 manifest and the
Phase 5 sensitivity artifacts (`reporting.run_phase6` fails closed otherwise).
No model selection or test-driven tuning happens here.

In [ ]:
import pandas as pd
from IPython.display import Image, display

from cheatdetect.eval import (
    plot_confusion_matrices,
    plot_pr_curves,
    plot_roc_curves,
    plot_score_distributions,
)
from cheatdetect.experiments.experiment_1 import reporting, runner
from cheatdetect.experiments.experiment_1.config import StudyConfig

config = StudyConfig()
study = runner.load_study(config)
run_dir = runner.DEFAULT_RUN_DIR
results, summary = reporting.run_phase6(study, run_dir=run_dir)
display(summary)

In [ ]:
# Scalar metrics per approach (primary + sensitivity), baseline included.
tables = pd.DataFrame(
    [{key: value for key, value in row.items() if key not in {"cm", "scores"}} for row in results]
)
display(tables)

## Grouped figures

Panels are grouped by model family and augmentation condition. Confusion
matrices use the seed-42 primary artifacts only and are never averaged across
seeds.

In [ ]:
figures = run_dir / reporting.REPORT_DIR_NAME / "figures"
figures.mkdir(parents=True, exist_ok=True)
y_test = study.raw["mixed_test"].labels
primary = [row for row in results if row["seed"] is None]

for family in ("if", "ocsvm", "ae", "ensemble"):
    for condition in ("aug_off", "aug_on"):
        group = [
            row
            for row in primary
            if row["family"] == family and row["augmented"] == (condition == "aug_on")
        ]
        if not group:
            continue
        scores = {row["model"]: row["scores"] for row in group}
        plot_roc_curves(scores, y_test, figures / f"roc_{family}_{condition}.png")
        plot_pr_curves(scores, y_test, figures / f"pr_{family}_{condition}.png")
        plot_score_distributions(
            scores, y_test, figures / f"scores_{family}_{condition}.png"
        )
        plot_confusion_matrices(
            group, figures / f"confusion_{family}_{condition}.png"
        )
        display(Image(filename=str(figures / f"roc_{family}_{condition}.png")))
        display(Image(filename=str(figures / f"confusion_{family}_{condition}.png")))
print(f"All figures written under {figures}")

## Seed sensitivity

Descriptive only: all three seeds, no best-seed selection.

In [ ]:
sensitivity = pd.DataFrame(
    [
        {
            "id": row["model"],
            "seed": row["seed"],
            "family": row["family"],
            "recipe": row["recipe"],
            "condition": "aug_on" if row["augmented"] else "aug_off",
            "roc_auc": row["roc_auc"],
            "pr_auc": row["pr_auc"],
            "threshold": row["threshold"],
        }
        for row in results
        if row["seed"] is not None
    ]
)
display(sensitivity)

## Search and training summary

Candidate counts, failures, runtime, and AE early-stopping status come from
the Phase 4 candidates table.

In [ ]:
candidates = pd.read_csv(run_dir / "candidates.csv")
display(
    candidates.groupby("family").agg(
        candidates=("status", "size"),
        failures=("status", lambda status: int((status == "failed").sum())),
        max_seconds=("seconds", "max"),
    )
)

ae_ok = candidates[(candidates["family"] == "ae") & (candidates["status"] == "ok")].copy()
ae_ok["hit_epoch_cap"] = ae_ok["epochs_trained"] >= config.lstm_epochs
display(
    ae_ok.groupby(["recipe", "augmented"]).agg(
        candidates=("status", "size"),
        epoch_cap_fraction=("hit_epoch_cap", "mean"),
        max_epochs=("epochs_trained", "max"),
        max_updates=("optimizer_updates", "max"),
    )
)